# 19 · 한 번에 — **4 seed 학습 → 500ep × 5회 eval**

한 노드에서 **seed 4개를 동시에 학습**하고(GPU 4장), 끝나면 그 자리에서
**150k 체크포인트를 5회 × 500 에피소드** 평가해 SR 표까지 낸다.

- 그룹/ task 는 아래 `TAGS` · `TASK` 로 고른다 (기본: `ours` · insertion).
- 개별 노트북(`01`~`08`, `12`~`17`)과 **같은 함수**를 부르므로 결과는 동일하다.
  이건 "학습→eval 을 한 창에서 쭉" 돌리고 싶을 때 쓰는 편의 노트북.
- 끊겨도 안전: 학습 = `--resume`, eval = 끝난 run 자동 skip.

⚠️ GPU 가 4장 있는 노드에서 쓸 것. 2장짜리 노드는 `12a/12b` 처럼 쪼갠 노트북을 쓴다.
⚠️ 학습 전에 **parity 테스트**(`00_smoke` 또는 `python tests/test_acm_sscp_literal.py`).


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

# ── 무엇을 / 어디서 ──────────────────────────────────────────────────────────
TASK = cf.MAIN_SIM          # 'insertion' (긴 horizon).  짧은 앵커는 cf.SHORT_SIM ('transfer')
TAGS = cf.GROUP_OURS        # ['ours']
# TAGS = cf.GROUP_ACM       #   ['acm'] — 대조군(우리 주장의 분모)
# TAGS = cf.GROUP_OURS + cf.GROUP_ACM      #   둘 다 (8잡)
# TAGS = cf.GROUP_BASELINE  #   act·diffusion·smolvla·acm2 (16잡)
# TAGS = cf.GROUP_ABLATION  #   acm_carry·acm_bimamba·acm_s7 (12잡)

SEEDS = cf.MAIN_SEEDS       # [0,1,2,3] — 4 seed 동시
GPUS  = cf.v23.available_gpus()
REPS  = list(range(cf.EVAL_REPEATS))   # 5회
N_EP  = cf.EVAL_N_EP                   # 500 에피소드

print('task :', TASK, cf.v23.TASKS[TASK])
print('모델 :', TAGS, '| seeds:', SEEDS, '| GPU:', GPUS)
print('학습 : %s step, lr 고정, 학습중 eval %s' % (f'{cf.STEPS:,}', cf.v23.EVAL_FREQ))
print('eval : %s ckpt x %d rep x %d ep' % (f'{cf.CKPT_STEP:,}', len(REPS), N_EP))
print()
print('학습 잡  :', len(TAGS) * len(SEEDS))
print('eval run :', len(TAGS) * len(SEEDS) * len(REPS),
      f'(= 에피소드 {len(TAGS)*len(SEEDS)*len(REPS)*N_EP:,})')

## 커맨드 확인 (dry-run)

In [ ]:
for t in TAGS:
    c = cf.make_train_cmd(t, seed=SEEDS[0], task=TASK, gpu_id=GPUS[0])
    print(f'{t:<10}', ' '.join(p for p in c.split()
                               if p.startswith(('CUDA_VISIBLE_DEVICES', '--dataset.repo_id',
                                                '--env.task', '--steps', '--policy.optimizer_lr'))))

## 1) 학습 — 4 seed 동시 (resume 자동)
첫 실행은 데이터셋을 한 번 먼저 받는다(prefetch). 안 그러면 잡들이 같은 HF 캐시에
동시 다운로드를 걸어 대부분 죽는다.

In [ ]:
jobs = cf.run_training(TAGS, SEEDS, task=TASK, gpus=GPUS)

## 2) 150k 체크포인트 확인 — 여기서 X 가 있으면 eval 하지 말 것

In [ ]:
ok = cf.print_ckpt_status(TAGS, SEEDS, TASK)
print('\n=>', 'eval 진행 가능' if ok else '⚠️ 학습 안 끝난 것이 있다')

## 3) eval — 150k 체크포인트 × 5회 × 500 에피소드
rep 마다 `--seed = 1000 + 100·rep` → env 초기상태가 달라진다(같은 seed 로 5번 돌리면 결정적이라 무의미).
학습 seed(모델 분산)와 rep(평가 분산)이 분리된다.

In [ ]:
cf.run_repeat_evals(TAGS, SEEDS, REPS, task=TASK, gpus=GPUS, n_episodes=N_EP)

## 4) 결과 — SR (mean ± std, 20 run) + pooled Wilson CI

In [ ]:
rows = cf.sr_table(TAGS, SEEDS, REPS, task=TASK, n_episodes=N_EP,
                   csv_path=cf.OUTPUT_BASE / 'main_report' / f'sr_150k_{TASK}.csv')

## 다음
- 다른 그룹도: 위 `TAGS` 만 바꿔서 이 노트북을 다시 실행 (`acm` → baseline → ablation 순 권장)
- 짧은 task: `TASK = cf.SHORT_SIM` (또는 2-GPU 노드에서 `12a`/`12b` …)
- 표·그림: `09_report_sr`(SR) · `10_report_jerk`(떨림) · `18_report_horizon`(SR vs horizon) · `11_efficiency`
